### Imports and R Environment Setup


In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from rpy2.robjects.conversion import localconverter

# Your updated package imports
from graphical_sampling.sampling import KMeansSampler
from graphical_sampling.population import Population
from package_sampling.utils import inclusion_probabilities

# Initialize the converter
numpy2ri_converter = numpy2ri.converter
conv = ro.default_converter + numpy2ri_converter

# Initialize R with updated scoring functions
ro.r("""
    library(BalancedSampling)
    library(sampling)
    library(WaveSampling)

    calc_r_metrics <- function(coords, probs, sample_idx) {
        coords_mat <- as.matrix(coords)
        probs_vec <- as.numeric(probs)
        
        # Note: indices in R are 1-based. 
        # sample_idx should be adjusted before passing or inside here.
        
        # Spatial Balance (SB)
        sb_val <- tryCatch(sb(probs_vec, coords_mat, sample_idx), error = function(e) Inf)
        
        # Spatial Balance Local (SBLB)
        sblb_val <- tryCatch(sblb(probs_vec, coords_mat, sample_idx), error = function(e) Inf)
        
        # Moran's I (IB) - requires weight matrix W
        W0 <- wpik(coords_mat, probs_vec)
        W <- W0 - diag(diag(W0))
        
        # Create a binary mask for the sample
        sample_mask <- rep(0, length(probs_vec))
        sample_mask[sample_idx] <- 1
        
        ib_val <- tryCatch(IB(W, sample_mask), error = function(e) Inf)
        
        return(c(ib = ib_val, sb = sb_val, sblb = sblb_val))
    }
""")

def evaluate_sampler(sampler: KMeansSampler):
    """
    Evaluates the KMeansSampler using both internal Python properties 
    and external R metrics.
    """
    # 1. Get all possible samples and their probabilities from the joint design
    all_samples = sampler.all_samples       # Array of shape (M, n)
    all_probs = sampler.all_samples_probs   # Array of shape (M,)
    
    results = []
    
    # Use tqdm for progress tracking
    for i in tqdm(range(len(all_samples)), desc="Evaluating Samples"):
        sample_indices = all_samples[i]
        # Adjust to 1-based indexing for R
        r_sample_idx = sample_indices + 1
        
        with localconverter(conv):
            # Pass data to R
            r_metrics = ro.r['calc_r_metrics'](
                sampler.coords, 
                sampler.probs, 
                r_sample_idx
            )
            r_metrics_np = np.array(r_metrics)
        
        # Python-side Density score
        # Using the internal cached property logic
        density_score = sampler.density_scores[i]
        
        results.append({
            'sample_id': i,
            'probability': all_probs[i],
            'density': density_score,
            'moran_i': r_metrics_np[0],
            'voronoi_sb': r_metrics_np[1],
            'local_balance': r_metrics_np[2]
        })
    
    df_results = pd.DataFrame(results)
    
    # Calculate Expected Values (Weighted Averages)
    summary = {
        'Exp_Density': np.sum(df_results['density'] * df_results['probability']),
        'Exp_Moran': np.sum(df_results['moran_i'] * df_results['probability']),
        'Exp_SB': np.sum(df_results['voronoi_sb'] * df_results['probability']),
        'Exp_LocalBalance': np.sum(df_results['local_balance'] * df_results['probability'])
    }
    
    return df_results, summary



### Optimized Sampling Functions

In [18]:
def run_sampling_design(method, coords, probs, n, num_samples, is_EP):
    N = len(coords)
    
    # 1. Python Methods
    if method == "Nmcs":
        # Create the updated Population object (handles normalization internally)
        pop = Population(coords=coords, probs=probs)
        
        # Initialize the updated KMeansSampler (split_size removed)
        # Note: adjust n_zones as needed for your specific experiment
        sampler = KMeansSampler(
            population=pop, 
            n=n, 
            n_zones=(2, 2), 
            zone_builder="sweep"
        )
        
        # Use the updated sample method
        return sampler.sample(num_samples)

    if method == "Rand":
        samples_idx = np.zeros((num_samples, n), dtype=int)
        for i in range(num_samples):
            samples_idx[i] = np.random.choice(N, n, replace=False)
        return samples_idx

    # 2. R Methods
    samples_idx = np.zeros((num_samples, n), dtype=int)
    
    with localconverter(conv):
        ro.globalenv['coords'] = coords
        ro.globalenv['probs'] = probs
        ro.globalenv['n'] = n
        
        # Prepare spatial features for R methods that require sf objects
        if method in ["HIP", "GRTS"]:
            ro.r("pts <- sf::st_as_sf(data.frame(x=coords[,1], y=coords[,2]), coords=c('x','y'))")
            if not is_EP: 
                ro.r("pts$probs <- probs")

        for i in range(num_samples):
            if method == "Lopi":
                # BalancedSampling::lpm2 returns 1-based indices
                samples_idx[i] = np.array(ro.r("BalancedSampling::lpm2(probs, coords)")) - 1
                
            elif method == "Wave":
                # Wave returns a binary mask
                mask = ro.r("WaveSampling::wave(coords, probs)")
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0]
                
            elif method == "Maxe":
                # UPmaxentropy returns a binary mask
                mask = ro.r("sampling::UPmaxentropy(probs)")
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0]
                
            elif method == "HIP":
                ro.r(f"res <- spbal::HIP(population = pts, n = {n})")
                # Extracting rownames and adjusting for 0-based indexing
                samples_idx[i] = np.array(ro.r("as.integer(rownames(res$sample))")) - 1
                
            elif method == "GRTS":
                # Conditional auxiliary variable for non-Equal Probability designs
                aux = ', n_prop = probs' if not is_EP else ""
                ro.r(f"res <- spsurvey::grts(sframe = pts, n_base = {n} {aux})")
                samples_idx[i] = np.array(ro.r("as.numeric(rownames(res$sites_base))")) - 1
                
    return samples_idx

def calculate_ht_estimator(y, sample_indices, probs):
    """
    Calculates the Horvitz-Thompson estimator for the population total.
    """
    # Ensure we handle potential -1 indices from empty clusters in Nmcs
    valid_mask = sample_indices >= 0
    valid_idx = sample_indices[valid_mask]
    
    sample_y = y[valid_idx]
    sample_probs = probs[valid_idx]
    
    # Total = sum(y_i / pi_i)
    return np.sum(sample_y / sample_probs)

### Metrics and Spread Calculation

In [19]:
def calculate_all_scores(coords, probs, sample_idx, n, N, density_measure, y_val):
    """
    Calculates various spatial and statistical scores for a given sample.
    
    Args:
        coords: Population coordinates.
        probs: Inclusion probabilities.
        sample_idx: 1D array of selected unit indices.
        n: Sample size.
        N: Population size.
        density_measure: An instance of the Density class.
        y_val: The variable of interest for HT estimation.
    """
    # 1. Density Score (Python)
    # The score method expects a 2D array of shape (n_samples, n)
    dens_score = density_measure.score(sample_idx.reshape(1, -1))
    
    # 2. HT Estimator Total
    # Uses the formula: Sum(y_i / pi_i)
    ht_val = np.sum(y_val[sample_idx] / probs[sample_idx])
    
    # 3. R Metrics (Spatial Balance)
    # Preparing data for the R environment
    with localconverter(conv):
        ro.globalenv['coords'] = coords
        ro.globalenv['probs'] = probs
        # R uses 1-based indexing for sample indices
        ro.globalenv['s_idx_r'] = sample_idx + 1 
        
        # Note: Your R function calc_r_metrics now takes (coords, probs, sample_idx)
        # We pass the R-adjusted 1-based indices.
        r_results = ro.r("calc_r_metrics(coords, probs, s_idx_r)")
        r_results_np = np.array(r_results)

    # Return Order: Density, Voronoi (sb), Moran (ib), Local Balance (sblb), HT_Total
    # r_results indices based on R function: c(ib, sb, sblb)
    return (
        dens_score[0],     # Density
        r_results_np[1],   # Voronoi (sb)
        r_results_np[0],   # Moran (ib)
        r_results_np[2],   # Local Balance (sblb)
        ht_val             # HT Estimator Total
    )

### The Main Execution Loop

In [20]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# Updated imports to match the provided files
from graphical_sampling.population import Population
from graphical_sampling.clustering import FIPBalancedNMeans
from graphical_sampling.index import Density

# --- Configuration ---
folder = "/config/ws/graphical-sampling/populations"
results_folder = "data_samples"
pop_names = ["random_uneq"]
sample_cnt = 10 
n_size = 5

os.makedirs(results_folder, exist_ok=True)

for name in pop_names:
    # 1. Load and Prep Data
    file_path = os.path.join(folder, f"{name}_N=100.csv")
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue
        
    df = pd.read_csv(file_path)
    coords = df[["x", "y"]].values.astype(float)
    
    # Calculate inclusion probabilities
    # Assuming inclusion_probabilities is a helper from your external package
    pik = inclusion_probabilities(df["prob"].values, n_size)
    n = int(np.round(np.sum(pik)))
    is_EP = np.allclose(pik, pik[0])

    print(f"\n--- Processing {name} (N={len(pik)}, n={n}) ---")

    # 2. Setup Density Measure using the updated Population class
    # The Population class handles internal normalization
    pop_wrapped = Population(coords=coords, probs=pik)
    
    # Initialize Density without split_size
    density_measure = Density(
        population=pop_wrapped, 
        k=n, 
        n_jobs=-1
    )

    # 3. Sampling and Scoring
    all_data = []
    # Added "Nmcs" to demonstrate usage of the new KMeansSampler logic
    methods = ["Rand", "Lopi", "Nmcs"] 

    for m in methods:
        print(f"Running {m}...")
        # Updated run_sampling_design handles the logic for different backends
        samples = run_sampling_design(m, coords, pik, n, sample_cnt, is_EP)
        
        for i in tqdm(range(sample_cnt), desc=f"Scoring {m}"):
            s_idx = samples[i]
            
            # Filter out invalid indices (e.g., -1 from empty clusters)
            valid_s_idx = s_idx[s_idx >= 0]
            
            # calculate_all_scores uses Density.score and R metrics
            metrics = calculate_all_scores(
                coords, 
                pik, 
                valid_s_idx, 
                n, 
                len(pik), 
                density_measure, 
                df["y"].values
            )
            
            all_data.append([m] + list(metrics))

    # 4. Results Processing
    if not all_data:
        continue

    res_df = pd.DataFrame(
        all_data, 
        columns=["Method", "Density", "Voronoi", "Moran", "Local_Balance", "HT_Total"]
    )
    
    # Group and calculate variance/mean for the HT Estimator
    summary = res_df.groupby("Method")["HT_Total"].agg(['var', 'mean'])
    
    # Calculate relative efficiency compared to Random Sampling
    if "Rand" in summary.index:
        summary["Efficiency"] = summary.loc["Rand", "var"] / summary["var"].replace(0, np.nan)

    print(summary)
    output_path = os.path.join(results_folder, f"final_results_{name}.csv")
    res_df.to_csv(output_path, index=False)


--- Processing random_uneq (N=100, n=5) ---
Running Rand...


Scoring Rand: 100%|██████████| 10/10 [00:00<00:00, 66.61it/s]


Running Lopi...


Scoring Lopi:   0%|          | 0/10 [00:00<?, ?it/s]

Scoring Lopi: 100%|██████████| 10/10 [00:00<00:00, 63.19it/s]


Running Nmcs...


Scoring Nmcs: 100%|██████████| 10/10 [00:00<00:00, 65.11it/s]

                var       mean  Efficiency
Method                                    
Lopi     106.496028  43.113325   19.265135
Nmcs     709.437658  53.877362    2.891953
Rand    2051.660371  83.789793    1.000000
